# ETL Transform: News → JSONL (GenAI)

This notebook demonstrates exporting transformed financial news from Postgres to **JSONL on S3** for GenAI / RAG use cases.

It reuses the existing ETL pipeline and the helper `export_genai_to_s3_from_db` which:
- loads news from PostgreSQL for a given date
- runs the standard text transformations (sentiment, intent, keywords, tickers)
- optionally generates embeddings
- writes partitioned JSONL files to S3 under paths like
  `news/transformed/crypto/agentic=true/year=YYYY/month=MM/day=DD/format=jsonl/news_transformed_yYYYY_mMM_dDD.jsonl`

Configure AWS and database credentials in your `.env` the same way as for `ETL-Transform-Text.ipynb`. Set `AWS_NEWS_BUCKET` or `AWS_DEFAULT_BUCKET` for S3 uploads. You can control which batch types are exported (run/week/month/year/day) via the `batch_types` argument.

In [1]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path('.').resolve()
project_root = _cwd
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent

src_path = project_root / 'src'
if not src_path.is_dir():
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from config.settings import get_settings
from pipelines.etl_cli import export_genai_to_s3_from_db

# --- Parameters (align with ETL-Transform-Text.ipynb) ---
USE_AGENTIC_ONLY = True  # Skip VADER/NLTK sentiment and use agentic-only enrichment
SINCE = "2026-07-13"
UNTIL = "2026-07-14"
INCLUDE_EMBEDDINGS = False
BATCH_TYPES = ['day']  # or a subset/superset, e.g. ['run', 'week', 'month', 'year', 'day']

# NOTE:
# - In agentic-only mode we ingest a date range via SINCE/UNTIL and enrich with the
#   FinancialMetricsTask, avoiding the nltk/VADER dependency.
# - For JSONL export, the helper uses SINCE (if provided) to drive the partition
#   keys (year/month/day) in the S3 paths, mirroring transformed CSV conventions.

_settings = get_settings()
_bucket = getattr(_settings.aws, "news_bucket", None) or _settings.aws.default_bucket
if not _bucket:
    print(
        "S3 upload skipped: set AWS_NEWS_BUCKET or AWS_DEFAULT_BUCKET in .env, "
        "then restart Jupyter."
    )
    uris = []
else:
    uris = export_genai_to_s3_from_db(
        since=SINCE,
        until=UNTIL,
        include_embeddings=INCLUDE_EMBEDDINGS,
        batch_types=BATCH_TYPES,
        use_agentic_only=USE_AGENTIC_ONLY,
    )
print('Exported GenAI JSONL URIs:')
for uri in uris:
    print('  ', uri)

Connection to the database successful!
Table name set to: financial_news_241118
Connection closed.
Agentic enrichment: processing 126 rows (daterange: 2026-07-13 to 2026-07-14, progress every 6 rows)
  progress: 6/126 rows
  progress: 12/126 rows
  progress: 18/126 rows
  progress: 24/126 rows
  progress: 30/126 rows
  progress: 36/126 rows
  progress: 42/126 rows
  progress: 48/126 rows
  progress: 54/126 rows
  progress: 60/126 rows
  progress: 66/126 rows
  progress: 72/126 rows
  progress: 78/126 rows
  progress: 84/126 rows
  progress: 90/126 rows
  progress: 96/126 rows
  progress: 102/126 rows
  progress: 108/126 rows
  progress: 114/126 rows
  progress: 120/126 rows
  progress: 126/126 rows
Agentic enrichment done: 126 rows
Exported GenAI JSONL URIs:
   s3://test-financial-news-bucket/news/transformed/crypto/agentic=true/year=2026/month=07/day=13/format=jsonl/news_transformed_y2026_m07_d13.jsonl
